# Task 02 – Model Training
## YOLOv8 Fine-Tuning for Drone Human & Car Detection

This notebook covers:
1. Model selection rationale
2. Dataset config setup
3. Training with augmentation
4. Loss & metric curves
5. Sample predictions

In [ ]:
import sys, os
sys.path.insert(0, '..')

import torch
import yaml
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

# ── Paths ──
DATA_ROOT   = r'D:\antlings-drone-detection\dataset\VisDrone_Dataset'
TRAIN_IMGS  = str(Path(DATA_ROOT) / 'VisDrone2019-DET-train' / 'images')
VAL_IMGS    = str(Path(DATA_ROOT) / 'VisDrone2019-DET-val'   / 'images')
TRAIN_LBLS  = str(Path(DATA_ROOT) / 'VisDrone2019-DET-train' / 'labels')
VAL_LBLS    = str(Path(DATA_ROOT) / 'VisDrone2019-DET-val'   / 'labels')

os.makedirs('../outputs/images', exist_ok=True)

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# Generate data.yaml pointing to the actual dataset location
data_yaml = {
    'path': DATA_ROOT,
    'train': 'VisDrone2019-DET-train/images',
    'val':   'VisDrone2019-DET-val/images',
    'test':  'VisDrone2019-DET-test-dev/images',
    'nc': 2,
    'names': {0: 'person', 1: 'car'}
}

os.makedirs('../configs', exist_ok=True)
with open('../configs/data.yaml', 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print('data.yaml written to configs/data.yaml')
print(yaml.dump(data_yaml))

## 1. Model Selection

| Model | mAP@.5 | Speed (FPS) | Params | Decision |
|-------|--------|-------------|--------|-----------|
| YOLOv8n | ~0.68 | ~42 | 3.2M | Selected |
| YOLOv8s | ~0.72 | ~28 | 11.2M | Alternative |
| RT-DETR | ~0.76 | ~15 | 42M | Transformer |
| Faster R-CNN | ~0.65 | ~10 | 41M | Two-stage |

**Why YOLOv8n:**
- Best speed/accuracy for real-time drone systems
- Built-in ByteTrack support
- Pretrained on COCO — already knows person and car
- Strong augmentation pipeline built in

In [ ]:
# First convert annotations (run once)
# This converts VisDrone format to YOLO format for train and val splits
import subprocess

result = subprocess.run(
    ['python', 'src/dataset.py', '--convert', '--data_root', DATA_ROOT],
    capture_output=True, text=True, cwd='..'
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[:500])

In [ ]:
# Load pretrained YOLOv8n
model = YOLO('yolov8n.pt')
print(f'Model loaded. Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ── TRAINING ──
# Set epochs=50 for full training, epochs=5 for quick demo

results = model.train(
    data='../configs/data.yaml',
    epochs=50,
    batch=16,           # reduce to 8 if GPU VRAM < 8GB
    imgsz=640,
    device=0 if torch.cuda.is_available() else 'cpu',

    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=3,

    # Augmentation — critical for aerial imagery
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.1,
    scale=0.5,
    flipud=0.3,
    degrees=10,

    conf=0.35,
    iou=0.45,

    project='../runs/detect',
    name='visdrone_yolov8n',
    plots=True,
)

BEST_WEIGHTS = f'{results.save_dir}/weights/best.pt'
print(f'\nBest weights saved to: {BEST_WEIGHTS}')

In [ ]:
# Plot training curves
import pandas as pd

results_csv = Path('../runs/detect/visdrone_yolov8n/results.csv')

if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle('Training Curves – YOLOv8n on VisDrone', fontsize=13, fontweight='bold')

    plots = [
        ('train/box_loss', 'val/box_loss', 'Box Loss',  axes[0,0]),
        ('train/cls_loss', 'val/cls_loss', 'Class Loss', axes[0,1]),
        ('train/dfl_loss', 'val/dfl_loss', 'DFL Loss',  axes[0,2]),
        ('metrics/precision(B)', None,     'Precision', axes[1,0]),
        ('metrics/recall(B)',    None,      'Recall',   axes[1,1]),
        ('metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'mAP', axes[1,2]),
    ]

    epochs = df['epoch'] if 'epoch' in df.columns else range(len(df))

    for train_col, val_col, title, ax in plots:
        if train_col in df.columns:
            ax.plot(epochs, df[train_col], label='train', color='#2979FF', linewidth=2)
        if val_col and val_col in df.columns:
            ax.plot(epochs, df[val_col], label='val', color='#FF6D00', linewidth=2)
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('../outputs/images/02_training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: outputs/images/02_training_curves.png')
else:
    print('results.csv not found — run training cell first.')

In [ ]:
# Show sample predictions on validation images
BEST_WEIGHTS = '../runs/detect/visdrone_yolov8n/weights/best.pt'
VAL_IMGS     = str(Path(DATA_ROOT) / 'VisDrone2019-DET-val' / 'images')

if Path(BEST_WEIGHTS).exists():
    model_best = YOLO(BEST_WEIGHTS)
    val_imgs   = sorted(Path(VAL_IMGS).glob('*.jpg'))[:6]

    fig, axes = plt.subplots(2, 3, figsize=(18, 11))
    fig.suptitle('Sample Predictions – Best Model on Validation Set', fontsize=13, fontweight='bold')

    CLASS_COLORS = {0: (0, 255, 80), 1: (0, 160, 255)}

    for idx, img_path in enumerate(val_imgs):
        frame  = cv2.imread(str(img_path))
        preds  = model_best.predict(frame, conf=0.35, iou=0.45, verbose=False)[0]
        ann    = frame.copy()
        p_cnt  = c_cnt = 0

        for box in preds.boxes:
            cls_id = int(box.cls[0])
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            cv2.rectangle(ann, (x1,y1), (x2,y2), CLASS_COLORS.get(cls_id,(255,255,255)), 2)
            if cls_id == 0: p_cnt += 1
            else: c_cnt += 1

        ax = axes[idx // 3][idx % 3]
        ax.imshow(cv2.cvtColor(ann, cv2.COLOR_BGR2RGB))
        ax.set_title(f'Person: {p_cnt}  Car: {c_cnt}', fontsize=11)
        ax.axis('off')

    plt.tight_layout()
    plt.savefig('../outputs/images/02_sample_predictions.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f'Weights not found at {BEST_WEIGHTS} — run training first.')

### Task 02 Complete

- Model: YOLOv8n fine-tuned from COCO pretrained weights
- Dataset: VisDrone_Dataset train/val splits
- 50 epochs, AdamW, cosine LR decay
- Best weights: `runs/detect/visdrone_yolov8n/weights/best.pt`